In [ ]:
import pandas as pd
import requests, zipfile, io, os

GTFS_URL = "https://bkk.hu/gtfs/budapest_gtfs.zip"
DATA_DIR = "bkk_data"
if not os.path.exists(DATA_DIR): os.makedirs(DATA_DIR)

print("Downloading BKK Data...")
r = requests.get(GTFS_URL)
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall(DATA_DIR, members=['stops.txt', 'stop_times.txt', 'routes.txt', 'trips.txt'])

stops = pd.read_csv(f"{DATA_DIR}/stops.txt")
stops['is_acc'] = stops['wheelchair_boarding'] == 1
stops['station_id'] = stops['parent_station'].fillna(stops['stop_id'])

stations = stops.groupby('station_id').agg({
    'stop_name': 'first',
    'stop_lat': 'mean',
    'stop_lon': 'mean',
    'is_acc': 'any'
}).reset_index()

st = pd.read_csv(f"{DATA_DIR}/stop_times.txt", usecols=['trip_id', 'stop_id', 'stop_sequence'])
stop_to_station = stops.set_index('stop_id')['station_id'].to_dict()
st['sid'] = st['stop_id'].map(stop_to_station)

st = st.sort_values(['trip_id', 'stop_sequence'])
st['next_sid'] = st.groupby('trip_id')['sid'].shift(-1)
edges = st.dropna(subset=['next_sid'])[['sid', 'next_sid']].drop_duplicates()

stations.to_pickle(f"{DATA_DIR}/stations.pkl")
edges.to_pickle(f"{DATA_DIR}/edges.pkl")
print(f"Saved {len(stations)} stations and {len(edges)} unique edges.")

Saved 5660 stations and 6827 unique edges.
